# 순환 신경망 기반 감성 분석 (알라딘 도서 리뷰)

네이버 영화 리뷰 실습(노트북 09)을 토대로, 직접 크롤링한 **알라딘 도서 리뷰**를 학습한다.

- 영화 리뷰: 긍정/부정 **2개 클래스**
- 알라딘 리뷰: 평점(rating)을 기준으로 **3개 클래스**로 구성
    - 평점 1~2 → `0` 부정
    - 평점 3   → `1` 중립
    - 평점 4~5 → `2` 긍정

진행 순서
1. 데이터 준비
2. 모델 구조 및 학습 설계 (구축)
3. 모델 학습
4. 모델 평가
5. 예측
6. 배포 (저장, 재사용)

## 1. 데이터 준비
    1-1. 데이터 로딩
    1-2. 데이터 전처리 (참고: 이미 완료된 _ing 파일 제공)
    1-3. 데이터 분리
    1-4. 학습용 데이터 준비
    1-5. 테스트용 데이터 준비

### 1-1. 데이터 로딩

크롤링 원본은 전처리(한글 정제 → 중복 제거 → Okt 형태소 토큰화 → 클래스 균형 샘플링)를 거쳐 
`./data/aladin_review_ing.csv` 로 저장해 두었다. (영화 실습의 `naver_movie_review_ing.csv` 와 동일한 형식)

- `clean_review` : 한글/공백만 남긴 정제 문장
- `tokens` : Okt 형태소 리스트
- `tokens_str` : 형태소를 공백으로 이어붙인 문자열 (← 신경망 입력으로 사용)
- `label` : 0(부정) / 1(중립) / 2(긍정)

In [ ]:
# 전처리 완료 파일 로딩
import pandas as pd
datafile = './data/aladin_review_ing.csv'
review_df = pd.read_csv(datafile, index_col=0)
review_df.head()

In [ ]:
review_df.info()

In [ ]:
# 클래스(label) 분포 확인 - 0:부정 / 1:중립 / 2:긍정
review_df.label.value_counts()

### 1-2. (참고) 크롤링 원본 → 전처리 과정

아래 셀은 위 `aladin_review_ing.csv` 를 **어떻게 만들었는지** 보여주는 참고용이다. 
원본 크롤링 데이터(수백만 건)에 Okt 토큰화를 돌리면 시간이 매우 오래 걸리므로 기본값은 실행하지 않는다(`RUN_PREPROCESS = False`). 
이미 전처리 결과가 있으므로 그대로 1-3 으로 진행하면 된다.

In [ ]:
RUN_PREPROCESS = False   # True 로 바꾸면 원본 크롤링 데이터에서 다시 전처리

if RUN_PREPROCESS:
    import re, glob
    from konlpy.tag import Okt
    from tqdm import tqdm
    tqdm.pandas()

    # (1) 크롤링 원본 로딩 : book_title, review_content, rating
    raw_path = r'C:\Users\user\Desktop\크롤링 내용\aladin_reviews_clean_3col.csv'
    raw_df = pd.read_csv(raw_path, encoding='utf-8-sig')

    # (2) 평점 → 레이블 (1~2:부정0 / 3:중립1 / 4~5:긍정2)
    raw_df['rating'] = pd.to_numeric(raw_df['rating'], errors='coerce')
    raw_df = raw_df[raw_df['rating'].between(1, 5)]
    raw_df['label'] = raw_df['rating'].apply(lambda r: 0 if r <= 2 else (1 if r == 3 else 2))

    # (3) 결측 제거
    raw_df = raw_df.dropna(subset=['review_content'])

    # (4) 정제 : 한글과 공백만 남기기
    raw_df['clean_review'] = raw_df.review_content.apply(lambda x: re.sub('[^ 가-힣]+', ' ', str(x)))
    raw_df.clean_review = raw_df.clean_review.apply(lambda x: re.sub('^ +', '', x))
    raw_df.clean_review = raw_df.clean_review.replace('', None)
    raw_df = raw_df.dropna(subset=['clean_review'])

    # (5) 중복 제거
    raw_df = raw_df.drop_duplicates(subset=['clean_review'])

    # (6) 클래스 균형 샘플링 (가장 적은 클래스 수에 맞춤)
    n = raw_df['label'].value_counts().min()
    raw_df = raw_df.groupby('label', group_keys=False).sample(n=n, random_state=42)

    # (7) Okt 형태소 토큰화
    raw_df['tokens'] = raw_df.clean_review.progress_apply(Okt().morphs)
    raw_df['tokens_str'] = raw_df.tokens.apply(lambda x: ' '.join(x))

    # (8) 저장
    raw_df.reset_index(drop=True).to_csv('./data/aladin_review_ing.csv')
    print('전처리 완료 →', raw_df.shape)
    review_df = raw_df.reset_index(drop=True)
else:
    print('전처리 생략: 기존 aladin_review_ing.csv 사용')

### 1-3. 데이터 분리
* 정답 데이터의 분포 확인 → 학습/테스트 분리 시 비율 유지 (`stratify`)

In [ ]:
# tokens_str 결측 제거 (혹시 모를 빈 값)
review_df = review_df.dropna(subset=['tokens_str'])

# 입력 데이터와 정답 데이터 추출 (list)
review_list = list(review_df.tokens_str)
label_list = list(review_df.label)
len(review_list), len(label_list)

In [ ]:
# label 컬럼 값별 데이터 수 (막대그래프)
review_df.label.value_counts().sort_index().plot(kind='bar')

In [ ]:
# 학습 데이터와 테스트 데이터 분리 (비율 유지)
from sklearn.model_selection import train_test_split

review_train, review_test, label_train, label_test = train_test_split(
    review_list, label_list, test_size=0.1, stratify=label_list, random_state=42)
len(review_train), len(review_test), len(label_train), len(label_test)

### 1-4. 학습 데이터 준비 (for tensorflow)
    1-4-1. Integer Encoding을 위한 tokenizer 생성
    1-4-2. 입력 데이터 Integer Encoding
    1-4-3. 입력 데이터 Padding
    1-4-4. 정답 데이터 원핫인코딩

#### 1-4-1. Integer Encoding을 위한 tokenizer 생성
* num_words = 사용할 단어 수(vocab_size) + 1 (0은 OOV/패딩에 할당)

In [ ]:
# 단어 수 제한 없이 Tokenizer 생성하여 단어 수 확인
from tensorflow.keras.preprocessing.text import Tokenizer
test_tokenizer = Tokenizer()
test_tokenizer.fit_on_texts(review_train)
list(test_tokenizer.word_index.items())[:10]

In [ ]:
from lib.my_utils import word_status_below_threshold
# 등장 빈도수를 threshold로 설정하여 버릴 단어가 차지하는 비율 확인
threshold = 3
word_status_below_threshold(test_tokenizer, threshold)

In [ ]:
# 단어 수를 제한하여 tokenizer 생성
# (도서 리뷰는 어휘가 다양하므로 단어 수를 넉넉히 잡는다. 위 비율을 보고 조정 가능)
vocab_size = 40000
num_words = vocab_size + 1
tokenizer = Tokenizer(num_words=num_words)
tokenizer.fit_on_texts(review_train)
len(tokenizer.word_index)

#### 1-4-2. 입력 데이터 Integer Encoding
* 제한된 단어에만 index를 부여하므로, 희귀 단어로만 구성된 리뷰는 길이가 0이 됨 → 제거

In [ ]:
# 입력 데이터 Integer Encoding
encoded_review_train = tokenizer.texts_to_sequences(review_train)
print(encoded_review_train[:5])

In [ ]:
# 길이가 0인 리뷰의 index 추출
null_index = [index for index, review in enumerate(encoded_review_train) if len(review) < 1]
len(null_index)

In [ ]:
# 길이가 1 이상인 리뷰로 학습 데이터 재구성
new_review_train = [review for index, review in enumerate(encoded_review_train) if index not in null_index]
new_label_train = [label for index, label in enumerate(label_train) if index not in null_index]
len(new_review_train), len(new_label_train)

#### 1-4-3. 입력 데이터 padding
* 입력 데이터의 길이(max_len)를 정하여 padding

In [ ]:
# 리뷰 길이 분포 확인 (히스토그램)
len_df = pd.DataFrame([len(review) for review in new_review_train])
len_df.hist()

In [ ]:
# 최대, 최소, 평균 등 정보 확인
len_df.describe()

In [ ]:
from lib.my_utils import text_len_status_below_maxlen
# 길이가 max_len 이하인 데이터의 비중 확인 (도서 리뷰는 영화평보다 길어서 넉넉히 설정)
max_len = 100
text_len_status_below_maxlen(new_review_train, max_len)

In [ ]:
# max_len 길이로 입력 데이터 padding
from tensorflow.keras.preprocessing.sequence import pad_sequences

train_X = pad_sequences(new_review_train, maxlen=max_len)
len(train_X), train_X[:2]

#### 1-4-4. 정답 데이터 one-hot encoding
* 3개 클래스이므로 to_categorical 결과는 3열

In [ ]:
from tensorflow.keras.utils import to_categorical

train_y = to_categorical(new_label_train, num_classes=3)
len(train_y), train_y[:5]

### 1-5. 테스트 데이터 준비
    1-5-1. 입력 데이터 Integer Encoding (결측 제거)
    1-5-2. 입력 데이터 padding
    1-5-3. 정답 데이터 one-hot encoding

In [ ]:
# 입력 데이터 Integer Encoding
encoded_review_test = tokenizer.texts_to_sequences(review_test)
print(encoded_review_test[:2])

In [ ]:
# 길이가 0인 리뷰 제거
null_index = [index for index, review in enumerate(encoded_review_test) if len(review) == 0]
new_review_test = [review for index, review in enumerate(encoded_review_test) if index not in null_index]
new_label_test = [label for index, label in enumerate(label_test) if index not in null_index]
len(new_review_test), len(new_label_test)

In [ ]:
# 입력 데이터 padding
test_X = pad_sequences(new_review_test, maxlen=max_len)
len(test_X), test_X[:2]

In [ ]:
# 정답 데이터 one-hot encoding
test_y = to_categorical(new_label_test, num_classes=3)
len(test_y), test_y[:5]

## 2. 모델 구축 및 컴파일
* 영화 실습과 달리 **3개 클래스**이므로 출력층 units=3, 손실함수는 `categorical_crossentropy`

In [ ]:
# 신경망 구조 설계
from tensorflow.keras.layers import Embedding, LSTM, Dense

input_units = num_words   # 사용한 feature 수 = 단어 수
embedding_dim = 32
lstm_units = 64
dense_units = 16
output_units = 3          # 분류 클래스 수: 부정/중립/긍정

rnn_model = [
    Embedding(input_units, embedding_dim),
    LSTM(lstm_units),
    Dense(dense_units, activation='tanh'),
    Dense(output_units, activation='softmax')
]

In [ ]:
# 신경망 구조 생성
from tensorflow.keras.models import Sequential

model = Sequential(rnn_model)
model.build(input_shape=(None, max_len))
model.summary()

In [ ]:
# 모델 학습 설계 (3개 클래스 → categorical_crossentropy)
from tensorflow.keras.optimizers import RMSprop
model.compile(loss='categorical_crossentropy', metrics=['accuracy'],
              optimizer=RMSprop(learning_rate=0.001))

In [ ]:
# EarlyStopping, ModelCheckpoint callback 설정
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

es = EarlyStopping(monitor='val_loss', mode='min', patience=3, verbose=1)
checkpoint_file = './model/best_model_aladin.keras'
mc = ModelCheckpoint(checkpoint_file, monitor='val_loss', mode='min', save_best_only=True)

## 3. 모델 학습

In [ ]:
# 모델 학습
history = model.fit(train_X, train_y, epochs=20, batch_size=128,
                    validation_split=0.1, callbacks=[es, mc])

In [ ]:
# 학습 곡선 시각화
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend(); plt.title('loss'); plt.show()

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend(); plt.title('accuracy'); plt.show()

## 4. 모델 평가

In [ ]:
# 저장된(가장 좋은) 가중치 로딩
model.load_weights(checkpoint_file)

In [ ]:
# 테스트 데이터로 평가
loss, acc = model.evaluate(test_X, test_y)
loss, acc

In [ ]:
# 예측값 구하기
import numpy as np
preds = model.predict(test_X)
result = [np.argmax(pred) for pred in preds]
result[:20]

In [ ]:
# 분류 리포트
from sklearn.metrics import classification_report
print(classification_report(new_label_test, result, target_names=['부정', '중립', '긍정']))

## 5. 예측

In [ ]:
# 입력된 리뷰에 대한 감성 판단 함수
from konlpy.tag import Okt
okt = Okt()

def analyze_sentiment(text):
    # 전처리 → 형태소 분석 → Integer Encoding → Padding
    tokens = okt.morphs(text)
    encoded_text = tokenizer.texts_to_sequences([tokens])
    X = pad_sequences(encoded_text, maxlen=max_len)
    preds = model.predict(X, verbose=0)
    labels = ['부정', '중립', '긍정']
    result_index = np.argmax(preds[0])
    return labels[result_index], preds[0][result_index]

In [ ]:
# 함수 테스트
reviews = [
    '내용이 알차고 정말 추천하고 싶은 책이에요',
    '돈이 아깝다 시간 낭비였습니다',
    '그냥 그저 그런 평범한 책',
    '번역이 어색하고 오타가 너무 많네요',
    '인생책이에요 여러 번 다시 읽었습니다',
    '무난하게 읽기 좋은 정도'
]

for review in reviews:
    result, prob = analyze_sentiment(review)
    print(f'{review} --> {result}({prob*100:.2f}%)')

## 6. 배포 (모델 저장 및 재사용)
    6-1. 모델 저장
    6-2. SentimentAnalyzer 클래스 구현 (저장된 모델 로딩 및 사용)

### 6-1. 모델 저장

In [ ]:
# keras 학습 모델 저장
model.save('./model/sa_model_aladin.keras')

In [ ]:
# Integer Encoding을 위한 tokenizer 직렬화
import joblib
joblib.dump(tokenizer, './model/sa_tokenizer_aladin.pkl')

### 6-2. SentimentAnalyzer 클래스 구현
- 객체 생성 시 예측 모델, Integer Encoder(tokenizer) 로딩
- 한국어 형태소 분석기 정의
- 입력된 리뷰의 감성 판단 함수 (부정/중립/긍정)

In [ ]:
from konlpy.tag import Okt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import joblib
import numpy as np

class SentimentAnalyzer:
    LABELS = ['부정', '중립', '긍정']

    def __init__(self, model_file, tokenizer_file, max_len=100):
        try:
            self.model = load_model(model_file)
        except (FileNotFoundError, OSError):
            print(f'모델 파일 {model_file} 이 없습니다.')
        try:
            self.tokenizer = joblib.load(tokenizer_file)
        except (FileNotFoundError, OSError):
            print(f'토크나이저 파일 {tokenizer_file} 이 없습니다.')
        self.max_len = max_len
        self.morphs = Okt().morphs

    def analyze_sentiment(self, text):
        # 전처리 → 형태소 분석 → Integer Encoding → Padding → 예측
        tokens = self.morphs(text)
        encoded_text = self.tokenizer.texts_to_sequences([tokens])
        X = pad_sequences(encoded_text, maxlen=self.max_len)
        preds = self.model.predict(X, verbose=0)
        result_index = np.argmax(preds[0])
        return self.LABELS[result_index], float(preds[0][result_index])

In [ ]:
# 저장한 모델 재사용 테스트
analyzer = SentimentAnalyzer(
    './model/sa_model_aladin.keras',
    './model/sa_tokenizer_aladin.pkl',
    max_len=100)

for review in reviews:
    result, prob = analyzer.analyze_sentiment(review)
    print(f'{review} --> {result}({prob*100:.2f}%)')